In [1]:
!pip install seqeval
# imports
import os
import re
import logging
import pandas as pd
from pprint import pprint
import numpy as np
from seqeval.metrics import (precision_score,
                             recall_score,
                             f1_score,
                             classification_report)

from datasets import (load_dataset,
                      DatasetDict, 
                      Features, 
                      Sequence, 
                      ClassLabel, 
                      Value, 
                      interleave_datasets, 
                      get_dataset_config_names, 
                      load_dataset, 
                      load_from_disk
)

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed
)

from ner_utility_functions.ner_utility_functions import (split_sources,
                            iter_entities,
                            trim_spans,
                            make_to_features_offset, 
                           
)
print("Imports ok")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=ef577d09217a77ab5fb140ea204ff9094a207bac97cd8398536dde155edb93ec
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


2026-08-15 14:18:43.042986: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786803523.268825      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786803523.325832      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786803523.852304      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786803523.852342      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786803523.852344      58 computation_placer.cc:177] computation placer alr

Imports ok


## 1. Data Loading and Exploration

In [2]:
# We list all available configurations of the dataset:
# configs = get_dataset_config_names("bigbio/swedish_medical_ner")
configs = get_dataset_config_names("community-datasets/swedish_medical_ner")
print("Available configurations:")
for config in configs:
    print(f"- {config}")

README.md: 0.00B [00:00, ?B/s]

Available configurations:
- 1177
- lt
- wiki


In [3]:
# Load dataset with all the chosen configurations

kb_datasets =[]
for config in configs:
    print(f"Attempting to load from: data/swedish_medical_ner_{config}")
    if os.path.isdir(f"data/swedish_medical_ner_{config}/train"):
        print(f"Loading configuration: {config} from disk")
        try:
            ds = load_from_disk(f"data/swedish_medical_ner_{config}")
            ds.config_name = config  # Attach config name as an attribute for later access.
            kb_datasets.append(ds)
            pprint(ds["train"].features) #Display schema  
        except Exception as e:
            print(f"Failed to load dataset from disk for config {config}: {e}")
        continue

    else:
        print(f"Loading configuration: {config} from the huggingface hub")
        ds = load_dataset("community-datasets/swedish_medical_ner", config)
        ds.config_name = config  # Attach config name as an attribute for later access.
        print(f"- {ds.config_name}")
        pprint(ds["train"].features) #Display schema
        kb_datasets.append(ds)
        #Saving to disk
        ds.save_to_disk(f"data/swedish_medical_ner_{ds.config_name}")

Attempting to load from: data/swedish_medical_ner_1177
Loading configuration: 1177 from the huggingface hub


1177/train-00000-of-00001.parquet:   0%|          | 0.00/77.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/927 [00:00<?, ? examples/s]

- 1177
{'entities': {'end': List(Value('int32')),
              'start': List(Value('int32')),
              'text': List(Value('string')),
              'type': List(ClassLabel(names=['Disorder and Finding', 'Pharmaceutical Drug', 'Body Structure']))},
 'sentence': Value('string'),
 'sid': Value('string')}


Saving the dataset (0/1 shards):   0%|          | 0/927 [00:00<?, ? examples/s]

...

Attempting to load from: data/swedish_medical_ner_wiki
Loading configuration: wiki from the huggingface hub


wiki/train-00000-of-00001.parquet:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48720 [00:00<?, ? examples/s]

- wiki
{'entities': {'end': List(Value('int32')),
              'start': List(Value('int32')),
              'text': List(Value('string')),
              'type': List(ClassLabel(names=['Disorder and Finding', 'Pharmaceutical Drug', 'Body Structure']))},
 'sentence': Value('string'),
 'sid': Value('string')}


Saving the dataset (0/1 shards):   0%|          | 0/48720 [00:00<?, ? examples/s]

In [4]:
# We extract entity types directly from the schema

#  We use the first loaded dataset, they all have the same schema
raw_ds = kb_datasets[0]  # or load fresh: load_dataset("community-datasets/swedish_medical_ner", "1177")

# Navigate the actual schema: entities is a dict of Lists
type_names = raw_ds["train"].features["entities"]["type"].feature.names
print(f"Entity types from dataset schema: {type_names}")

# Build BIO labels
def normalize_type_name(name):
    """Convert 'Disorder and Finding' -> 'disorder_finding'"""
    return name.lower().replace(" and ", "_").replace(" ", "_")

label_list = ["O"]
for type_name in sorted(type_names, key=normalize_type_name):
    token = normalize_type_name(type_name)
    label_list.append(f"B-{token}")
    label_list.append(f"I-{token}")

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"label_list: {label_list}")
print(f"label2id: {label2id}")


Entity types from dataset schema: ['Disorder and Finding', 'Pharmaceutical Drug', 'Body Structure']
label_list: ['O', 'B-body_structure', 'I-body_structure', 'B-disorder_finding', 'I-disorder_finding', 'B-pharmaceutical_drug', 'I-pharmaceutical_drug']
label2id: {'O': 0, 'B-body_structure': 1, 'I-body_structure': 2, 'B-disorder_finding': 3, 'I-disorder_finding': 4, 'B-pharmaceutical_drug': 5, 'I-pharmaceutical_drug': 6}


In [5]:
# The three configurations are explored

pd.set_option('display.max_colwidth', None)

for i, dataset in enumerate(kb_datasets):
    print(f"### Configuration {i + 1}: {dataset.config_name}")
    print(f"Rows: {dataset['train'].num_rows}")  # Shows splits and number of examples
    print(f"Columns: {len(dataset['train'].features)}")  # Number of columns/features

    # Convert a slice of the dataset to a pandas dataframe and display it
    example = dataset["train"].select(range(10)).to_pandas()
    display(example)

### Configuration 1: 1177
Rows: 927
Columns: 3


,sid,sentence,entities
0,1177_0,Memantin ( Ebixa ) ger sällan några biverkningar.,"{'start': [9], 'end': [18], 'text': ['Ebixa'], 'type': [0]}"
1,1177_1,Det är också lättare att dosera [ flytande medicin ] än att dela på tabletter.,"{'start': [32], 'end': [52], 'text': ['flytande medicin'], 'type': [1]}"
2,1177_2,( Förstoppning ) är ett vanligt problem hos äldre.,"{'start': [0], 'end': [16], 'text': ['Förstoppning'], 'type': [0]}"
3,1177_3,[ Medicinen ] kan också göra att man blöder lättare eftersom den påverkar { blodets } förmåga att levra sig.,"{'start': [0, 74], 'end': [13, 85], 'text': ['Medicinen', 'blodets'], 'type': [1, 2]}"
4,1177_4,Barn har större möjligheter att samarbeta om de i förväg får veta vad som ska hända.,"{'start': [], 'end': [], 'text': [], 'type': []}"
5,1177_5,Eftersom de påverkar hela kroppen mer än övriga mediciner bör man bara ta dem när olika kombinationer av receptfria mediciner inte hjälper.,"{'start': [], 'end': [], 'text': [], 'type': []}"
6,1177_6,För att få ett skydd mot ( hepatit B ) behövs tre doser vaccin.,"{'start': [25], 'end': [38], 'text': ['hepatit B'], 'type': [0]}"
7,1177_7,Effekten av naproxen sitter i längre och varar cirka 12 timmar jämfört med cirka 6 timmar för ibuprofen.,"{'start': [], 'end': [], 'text': [], 'type': []}"
8,1177_8,[ Cox-hämmare ] finns även som gel och sprej.,"{'start': [0], 'end': [15], 'text': ['Cox-hämmare'], 'type': [1]}"
9,1177_9,"Det är bra om ett litet barn är mätt och utsövt, eftersom de flesta påfrestningar då känns mindre.","{'start': [], 'end': [], 'text': [], 'type': []}"


### Configuration 2: lt
Rows: 745753
Columns: 3


,sid,sentence,entities
0,lt_0,", (hjärtinfarkt) och (syndrom) som vi nu år 1999 inte ens vet na","{'start': [2, 21], 'end': [16, 30], 'text': ['hjärtinfarkt', 'syndrom'], 'type': [0, 0]}"
1,lt_1,"tinernas goda effekt på morbiditeten är välkänd, och data hi","{'start': [], 'end': [], 'text': [], 'type': []}"
2,lt_2,"[sukralfat], [lakrits] och vismut) som kunde utgöra ett skydd öv","{'start': [0, 13], 'end': [11, 22], 'text': ['sukralfat', 'lakrits'], 'type': [1, 1]}"
3,lt_3,och tveksamhet {vad} gäller operationsindikationen kan man ha,"{'start': [16], 'end': [21], 'text': ['vad'], 'type': [2]}"
4,lt_4,1989 blev en anmälningspliktig (sjukdom) enligt Smittskyddsla,"{'start': [32], 'end': [41], 'text': ['sjukdom'], 'type': [0]}"
5,lt_5,kombinerat med remodellering av (hjärtat). Detta säkras genom,"{'start': [32], 'end': [41], 'text': ['hjärtat'], 'type': [0]}"
6,lt_6,olyckshändelse radikalt förändrat deras liv. {Sigmoideum} är,"{'start': [46], 'end': [58], 'text': ['Sigmoideum'], 'type': [2]}"
7,lt_7,ra att hon samtidigt ordinerade [Cyklokapron] i en mängd av 5,"{'start': [32], 'end': [45], 'text': ['Cyklokapron'], 'type': [1]}"
8,lt_8,till vara erfarenheterna och föra ut kunskapen till sjukvård,"{'start': [], 'end': [], 'text': [], 'type': []}"
9,lt_9,es kring behandling med betablockad vid (kronisk hjärtsvikt).,"{'start': [40], 'end': [60], 'text': ['kronisk hjärtsvikt'], 'type': [0]}"


### Configuration 3: wiki
Rows: 48720
Columns: 3


,sid,sentence,entities
0,wiki_0,"{kropp} beskrivs i till exempel människokroppen, anatomi och f","{'start': [0], 'end': [7], 'text': ['kropp'], 'type': [2]}"
1,wiki_1,"sju miljoner år gammalt hominint {kranium}, klassificerad som","{'start': [33], 'end': [42], 'text': ['kranium'], 'type': [2]}"
2,wiki_2,autosomer och ett par könskromosomer. Varje {kromosom} består,"{'start': [45], 'end': [55], 'text': ['kromosom'], 'type': [2]}"
3,wiki_3,{kromosom} består av en DNA-molekyl och {protein}. En DNA-molek,"{'start': [1], 'end': [50], 'text': ['kromosom} består av en DNA-molekyl och {protein'], 'type': [2]}"
4,wiki_4,tikel:Människans {skelett} Människans skelett är det skelett s,"{'start': [17], 'end': [26], 'text': ['skelett'], 'type': [2]}"
5,wiki_5,os människor. En vuxen människas {skelett} består av 206 till,"{'start': [33], 'end': [42], 'text': ['skelett'], 'type': [2]}"
6,wiki_6,"{lett} består av 206 till 220 {ben}, beroende på hur man räknar.","{'start': [0], 'end': [35], 'text': ['lett} består av 206 till 220 {ben'], 'type': [2]}"
7,wiki_7,v kroppsvikten.Ett nyfött barn har ca 300 {ben} i kroppen vilk,"{'start': [42], 'end': [47], 'text': ['ben'], 'type': [2]}"
8,wiki_8,kollektivet i mindre bitar såsom länder > städer > orter {Hud},"{'start': [57], 'end': [62], 'text': ['Hud'], 'type': [2]}"
9,wiki_9,sdjur. {Huden} utgör ett mekaniskt skydd mot omvärlden och bid,"{'start': [7], 'end': [14], 'text': ['Huden'], 'type': [2]}"


We have three configurations (subsets) with varying size

The data of interest are:
* Passage text (a full sentence (1177) or part of a sentence (wiki, lt)).
* The Named Entities (bracketed using: (), [] {}),
* their starting and ending positions: start, end,
* and their types (0,1 and 2).

The types refer to the the different types of named entities (we may also call them labels or classes):
* 'Pharmaceutical Drug': 0
* 'Disorder and Finding': 1
* 'Body Structure': 2


## The task: NER


The task of Named entity Recognition (NER) is one of sequence labeling, where each token in a sentence must be tagged.

The following sentence contains two Named Entities of different types (1,2):

`[ Medicinen ] kan också göra att man blöder lättare eftersom den påverkar { blodets } förmåga att levra sig.	{'start': [0, 74], 'end': [13, 85], 'text': ['Medicinen', 'blodets'], 'type': [1, 2]}`

As we can see, the raw data gives the named entities as *spans* with start/end positions.
The logical next step is to convert the spans to *per-token labels*, i e to associate each token within a sentence with it's corresponding type label.

First however, we split the configurations into training and validation sets

### 1) Split each config into train/val

In [6]:
per_source_raw= split_sources(kb_datasets, val_fraction=0.05, seed=42)
print(per_source_raw)

Map:   0%|          | 0/880 [00:00<?, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Map:   0%|          | 0/708465 [00:00<?, ? examples/s]

Map:   0%|          | 0/37288 [00:00<?, ? examples/s]

Map:   0%|          | 0/46284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2436 [00:00<?, ? examples/s]

{'1177': DatasetDict({
    train: Dataset({
        features: ['sid', 'sentence', 'entities', 'source'],
        num_rows: 880
    })
    validation: Dataset({
        features: ['sid', 'sentence', 'entities', 'source'],
        num_rows: 47
    })
}), 'lt': DatasetDict({
    train: Dataset({
        features: ['sid', 'sentence', 'entities', 'source'],
        num_rows: 708465
    })
    validation: Dataset({
        features: ['sid', 'sentence', 'entities', 'source'],
        num_rows: 37288
    })
}), 'wiki': DatasetDict({
    train: Dataset({
        features: ['sid', 'sentence', 'entities', 'source'],
        num_rows: 46284
    })
    validation: Dataset({
        features: ['sid', 'sentence', 'entities', 'source'],
        num_rows: 2436
    })
})}


### 2) Convert entities to list-of-dicts format.

We keep `type` as `int` `ClassLabel`in order to keep the downstream code cleaner, and to simplify iteration.

In [7]:
def dict_of_lists_to_list_of_dicts(entities_dict):
    """Convert a dictionary of lists to a list of dictionaries."""
    return [
        {"start": s, "end": e, "text": txt, "type": t}
        for s, e, txt, t in zip(
            entities_dict["start"],
            entities_dict["end"],
            entities_dict["text"],
            entities_dict["type"]
        )
    ]

# Apply to every split inalready-split dict: per_source_raw (1177/lt/wiki)
per_source_norm = {}
for cfg, ds in per_source_raw.items():
    per_source_norm[cfg] = ds.map(
        lambda ex: {
            **ex,
            "entities": dict_of_lists_to_list_of_dicts(ex["entities"])
        }
    )
# We inspect the first few examples 
#per_source_norm["1177"]["train"].select(range(3)).to_pandas()[["sid","sentence","entities"]]
#per_source_norm["1177"]["validation"].select(range(3)).to_pandas()[["sid","sentence","entities"]]


Map:   0%|          | 0/880 [00:00<?, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Map:   0%|          | 0/708465 [00:00<?, ? examples/s]

Map:   0%|          | 0/37288 [00:00<?, ? examples/s]

Map:   0%|          | 0/46284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2436 [00:00<?, ? examples/s]

In [8]:
# We do a check
for cfg, ds in per_source_norm.items():
    print(f"Config: {cfg}")
    for split in ds.keys():
        print(f"  Split: {split}, first example 'entities': {ds[split][0]['entities']}")
    print("---")


Config: 1177
  Split: train, first example 'entities': [{'end': 11, 'start': 0, 'text': 'Alvedon', 'type': 1}, {'end': 56, 'start': 46, 'text': 'munnen', 'type': 2}, {'end': 101, 'start': 70, 'text': 'munsönderfallande tabletter', 'type': 1}]
  Split: validation, first example 'entities': [{'end': 10, 'start': 0, 'text': 'Demens', 'type': 0}]
---
Config: lt
  Split: train, first example 'entities': [{'end': 20, 'start': 11, 'text': 'syndrom', 'type': 0}]
  Split: validation, first example 'entities': [{'end': 52, 'start': 41, 'text': 'läkemedel', 'type': 1}]
---
Config: wiki
  Split: train, first example 'entities': [{'end': 43, 'start': 34, 'text': 'lysosom', 'type': 2}]
  Split: validation, first example 'entities': [{'end': 32, 'start': 13, 'text': 'limbiska systemet', 'type': 2}]
---


We now have the entities as a list: `list[{"start","end","text","type"}]`, and the type as an `int` (ClassLabel id), matching the dataset’s schema.


### 3) Build global BIO labels (union over configs)
We'll keep the BIO tags readable (B-body_structure, etc.) by mapping the int codes to the official names during featurization. No dataset mutation needed.

In [9]:
# Diagnostic cell:
print(type(per_source_raw))
print(type(per_source_raw["1177"]))
print(type(per_source_raw["1177"]["train"]))
print(per_source_raw["1177"]["train"].features)

<class 'dict'>
<class 'datasets.dataset_dict.DatasetDict'>
<class 'datasets.arrow_dataset.Dataset'>
{'sid': Value('string'), 'sentence': Value('string'), 'entities': {'start': List(Value('int32')), 'end': List(Value('int32')), 'text': List(Value('string')), 'type': List(ClassLabel(names=['Disorder and Finding', 'Pharmaceutical Drug', 'Body Structure']))}, 'source': Value('string')}


In [10]:
print(per_source_raw["1177"]["train"].features["entities"])

{'start': List(Value('int32')), 'end': List(Value('int32')), 'text': List(Value('string')), 'type': List(ClassLabel(names=['Disorder and Finding', 'Pharmaceutical Drug', 'Body Structure']))}


### 4) Normalize entities (shape + types)

In [11]:
from transformers import AutoTokenizer
from datasets import DatasetDict

MODEL_NAME = "KB/bert-base-swedish-cased"  # Downloads base model without any classification head

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")
# Read names from RAW split (has the ClassLabel feature)
#type_names = per_source_raw["1177"]["validation"].features["entities"].feature["type"].names # Ad hoc fix

split = per_source_norm["1177"]["validation"]   # <- normalized
# ['Disorder and Finding', 'Pharmaceutical Drug', 'Body Structure']

to_features = make_to_features_offset(tokenizer, label2id, type_names, max_length=256)

ds_1177_val_feats = split.map(
    to_features,
    batched=False,
    remove_columns=split.column_names,
    desc="[1177] validation featurization",
)



config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at KB/bert-base-swedish-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer vocabulary size: 50325


[1177] validation featurization:   0%|          | 0/47 [00:00<?, ? examples/s]

In [12]:
NAME_TO_TOKEN = {
    "Disorder and Finding": "disorder_finding",
    "Pharmaceutical Drug": "pharmaceutical_drug",
    "Body Structure": "body_structure",
}

ex0 = per_source_norm["1177"]["validation"][2]
print(ex0["sentence"])
print("entities:", list(iter_entities(ex0, type_names, NAME_TO_TOKEN)))

enc0 = tokenizer(ex0["sentence"], return_offsets_mapping=True, truncation=True, max_length=256)
labs0 = make_to_features_offset(tokenizer, label2id, type_names)(ex0)["labels"]
tokens = tokenizer.convert_ids_to_tokens(enc0["input_ids"])

for tok, off, lab_id in zip(tokens, enc0["offset_mapping"], labs0):
    if lab_id == -100: 
        continue
    print(f"{tok:15} {off}  {id2label[lab_id]}")


Nysningar och ( nästäppa ) kan ofta dämpas av [ nässprej ] , kliande och ( svullna ögon ) går att behandla med [ ögondroppar ] .
entities: [(14, 26, 'disorder_finding'), (46, 58, 'pharmaceutical_drug'), (73, 89, 'disorder_finding'), (111, 126, 'pharmaceutical_drug')]
Ny              (0, 2)  O
##sn            (2, 4)  O
##ingar         (4, 9)  O
och             (10, 13)  O
(               (14, 15)  O
näst            (16, 20)  B-disorder_finding
##äpp           (20, 23)  I-disorder_finding
##a             (23, 24)  I-disorder_finding
)               (25, 26)  O
kan             (27, 30)  O
ofta            (31, 35)  O
dämpa           (36, 41)  O
##s             (41, 42)  O
av              (43, 45)  O
[               (46, 47)  O
näs             (48, 51)  B-pharmaceutical_drug
##spre          (51, 55)  I-pharmaceutical_drug
##j             (55, 56)  I-pharmaceutical_drug
]               (57, 58)  O
,               (59, 60)  O
kli             (61, 64)  O
##ande          (64, 68)  O
och        

Note that brackets `()`, `[]`, `{}` are now labeled as O. However, the start and end positions are the same as before pointing to the brackets. The BIO labels align with real entity content only, not the brackets. More importantly, the text has been further tokenized, using AutoTokenizer.

## Task 1: Survey of Language Models for Swedish Biomedical NER

### 1.1 Problem Context

Named Entity Recognition (NER) in Swedish biomedical text presents unique challenges. Swedish is a low-resource language compared to English, and clinical text contains specialized vocabulary, abbreviations, and domain-specific conventions that differ significantly from general-domain text (Almgren & Pavlov, 2016). The task requires identifying three entity types: **Disorder and Finding** (sjukdom & symtom), **Pharmaceutical Drug** (läkemedel), and **Body Structure** (kroppsdel).

Two fundamental approaches exist for this task:

1. **Fine-tune a Swedish language model** on biomedical NER data
2. **Translate Swedish text to English** and use domain-specific English models (e.g., BioBERT, Clinical-BERT)

### 1.2 Available Models

| Model              | Language                 | Domain     | Training Data                              | Availability            |
| ------------------ | ------------------------ | ---------- | ------------------------------------------ | ----------------------- |
| **KB-BERT**        | Swedish                  | General    | 18.3 GB (news, books, Wikipedia, gov docs) | ✅ Public (Hugging Face) |
| **M-BERT**         | Multilingual (104 langs) | General    | Wikipedia                                  | ✅ Public                |
| **SweDeClin-BERT** | Swedish                  | Clinical   | 17.9 GB EHRs (~2M records)                 | ❌ Restricted access     |
| **BioBERT**        | English                  | Biomedical | PubMed abstracts (21.3B words)             | ✅ Public                |
| **Clinical-BERT**  | English                  | Clinical   | MIMIC-III (112K EHRs)                      | ✅ Public                |

### 

### 1.3 Performance Comparison

Rosvall & Paasonen (2023) systematically compared these models on the Stockholm EPR PHI Corpus for Swedish clinical NER:

| Model                          | F1-Score (Full Data) | Notes                                            |
| ------------------------------ | -------------------- | ------------------------------------------------ |
| **SweDeClin-BERT**             | ~0.94                | Best overall, especially for small datasets      |
| **KB-BERT**                    | ~0.91                | Strong second, good generalization               |
| **M-BERT**                     | ~0.88                | Decent but underperforms Swedish-specific models |
| **BioBERT** (translated)       | ~0.82                | Translation errors degrade performance           |
| **Clinical-BERT** (translated) | ~0.80                | Domain match helps, but translation hurts        |

Key findings:

- **SweDeClin-BERT** excels due to pre-training on Swedish clinical text, capturing domain-specific terminology and abbreviations
- **KB-BERT** performs well despite being general-domain, benefiting from extensive Swedish pre-training
- **Translation-based approaches** (BioBERT, Clinical-BERT) suffer from translation artifacts, particularly for clinical abbreviations and compound words

### 1.4 Trade-off Analysis: Swedish LLM vs. Translation Approach

| Aspect                     | Swedish LLM (KB-BERT)                              | Translation + English Model                    |
| -------------------------- | -------------------------------------------------- | ---------------------------------------------- |
| **Linguistic fidelity**    | ✅ Native handling of Swedish morphology, compounds | ❌ Translation errors, especially abbreviations |
| **Domain knowledge**       | ⚠️ General domain (requires fine-tuning)            | ✅ Pre-trained on biomedical/clinical text      |
| **Pipeline complexity**    | ✅ Single model                                     | ❌ Translation + NER + back-mapping             |
| **Error propagation**      | ✅ Minimal                                          | ❌ Translation errors cascade to NER            |
| **Clinical abbreviations** | ✅ Can learn Swedish-specific forms                 | ❌ "SSK" → nurse fails to translate             |
| **Reproducibility**        | ✅ End-to-end                                       | ⚠️ Depends on translation model version         |

### 1.5 The SweDeClin-BERT Question

SweDeClin-BERT (Vakili et al.) achieves the highest reported performance on Swedish clinical NER. However, it is trained on sensitive EHR data from Swedish hospitals and is **not publicly available**. Access requires application to the Swedish Health Record Research Bank and institutional agreements, making it impractical for this project.

### 1.6 Summary

For Swedish biomedical NER, the evidence suggests:

1. **Swedish-specific models outperform multilingual models** — linguistic specialization matters
2. **Translation-based approaches introduce unacceptable error rates** — clinical abbreviations and terminology do not translate reliably
3. **Domain-specific pre-training helps significantly** — but requires access to restricted clinical data
4. **KB-BERT is the best publicly available option** — strong Swedish language understanding can compensate for lack of domain-specific pre-training through task-specific fine-tuning

------

## 

## Task 2: Model Selection and Justification

### 2.1 Selected Model: KB-BERT (`KB/bert-base-swedish-cased`)

We select **KB-BERT** (Malmsten et al., 2020) as our base model for Swedish biomedical NER.

### 2.2 Justification

**1. Best available Swedish language understanding**

KB-BERT was trained on 18.3 GB of Swedish text spanning diverse sources (news, literature, government documents, Wikipedia). This provides robust understanding of Swedish grammar, morphology, and vocabulary — a foundation that transfer learning can adapt to the biomedical domain.

Grancharova & Dalianis (2021) showed KB-BERT achieves F1=0.92 on Swedish clinical NER, outperforming M-BERT despite no clinical pre-training.

**2. Public availability and reproducibility**

Unlike SweDeClin-BERT (the theoretical best choice), KB-BERT is freely available on Hugging Face without access restrictions. This enables:

- Full reproducibility of experiments
- Use in downstream applications without licensing constraints
- Community building and comparison with other researchers

**3. Proven NER capability**

The KB team has released a general-domain NER variant (`KB/bert-base-swedish-cased-ner`) fine-tuned on SUC 3.0. While this targets different entity types (PER, LOC, ORG), it demonstrates the architecture's suitability for Swedish sequence labeling.

**4. Translation approach rejected**

We explicitly reject the BioBERT/Clinical-BERT translation approach because:

- Clinical abbreviations (e.g., "hö" for "höger", "SSK" for nurse) do not translate correctly
- Compound words are common in Swedish medical text and often mistranslated
- The translation pipeline adds complexity and potential failure points
- Rosvall & Paasonen (2023) found translated approaches underperform by ~10 F1 points

### 2.3 Limitations and Mitigations

| Limitation                    | Mitigation                                      |
| ----------------------------- | ----------------------------------------------- |
| No clinical pre-training      | Fine-tune on domain-specific data (1177 subset) |
| General vocabulary            | BIO tagging allows learning new entity spans    |
| Limited medical abbreviations | Training data from 1177.se uses full terms      |

### 2.4 Alternative Considered: Multilingual BERT

M-BERT was considered as a fallback option given its multilingual capabilities. However:

- It underperforms KB-BERT on Swedish tasks consistently
- The Swedish portion of its training data is limited to Wikipedia
- No benefit from cross-lingual transfer for our monolingual task


### 2.5 Conclusion

KB-BERT represents the optimal trade-off between performance, availability, and reproducibility for Swedish biomedical NER. While SweDeClin-BERT would theoretically perform better, its restricted access makes it unsuitable. Fine-tuning KB-BERT on the 1177 gold-standard subset should yield strong results for our clinical entity extraction task.

------

## 

## References

- Almgren, S., & Pavlov, S. (2016). *Named Entity Recognition in Swedish Medical Journals*. Chalmers University of Technology.
- Grancharova, M., & Dalianis, H. (2021). Comparing BERT Models for Clinical NER on Swedish Electronic Health Records. *CEUR Workshop Proceedings*.
- Malmsten, M., Börjeson, L., & Haffenden, C. (2020). Playing with Words at the National Library of Sweden. *arXiv:2007.01658*.
- Rosvall, E., & Paasonen, A. (2023). *Data Augmentation for Swedish Clinical NER*. Chalmers University of Technology.
- Vakili, T., et al. (2022). SweDeClin-BERT: A Swedish Clinical BERT Model. *Swedish Health Record Research Bank*.

## Task 3: Fine-tuning KB-BERT for Swedish Biomedical NER

In this section, we fine-tune KB-BERT on the 1177 subset of the Swedish Medical NER dataset. We use only the 1177 subset because:

1. **Gold-standard annotations**: The 1177 data was manually annotated by domain experts from Vårdguiden (Swedish healthcare portal)
2. **Data quality**: The `lt` and `wiki` subsets use distant supervision (automatic labeling), which introduces noise
3. **Clinical relevance**: 1177.se content mirrors the language used in patient-facing clinical documentation

### 3.1 Prepare Training Data

In [13]:
# Apply featurization to train and validation splits
# We use only the 1177 source (gold-standard annotations)

source_key = "1177"
ds_featurized = {}

for split_name in ["train", "validation"]:
    print(f"Featurizing {source_key}/{split_name}...")
    ds_featurized[split_name] = per_source_norm[source_key][split_name].map(
        to_features,
        batched=False,
        remove_columns=per_source_norm[source_key][split_name].column_names,
        desc=f"Tokenizing {split_name}"
    )
    print(f"  {len(ds_featurized[split_name])} examples")

print(f"\nDataset ready:")
print(f"  Train: {len(ds_featurized['train'])} examples")
print(f"  Validation: {len(ds_featurized['validation'])} examples")

Featurizing 1177/train...


Tokenizing train:   0%|          | 0/880 [00:00<?, ? examples/s]

  880 examples
Featurizing 1177/validation...


Tokenizing validation:   0%|          | 0/47 [00:00<?, ? examples/s]

  47 examples

Dataset ready:
  Train: 880 examples
  Validation: 47 examples


### 3.2 Load Model and Configure Training

We load KB-BERT and add a token classification head with our BIO label schema (7 labels: O + B/I for 3 entity types).

In [14]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import numpy as np

# Load KB-BERT with a token classification head
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

print(f"Model: {MODEL_NAME}")
print(f"Labels: {label_list}")
print(f"Parameters: {model.num_parameters():,}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at KB/bert-base-swedish-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: KB/bert-base-swedish-cased
Labels: ['O', 'B-body_structure', 'I-body_structure', 'B-disorder_finding', 'I-disorder_finding', 'B-pharmaceutical_drug', 'I-pharmaceutical_drug']
Parameters: 124,105,735


In [15]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_metrics(eval_pred):
    """Compute NER metrics using seqeval (entity-level evaluation)."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    
    # Convert IDs back to label strings, ignoring padding (-100)
    true_labels = []
    true_predictions = []
    
    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags = []
        gold_tags = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:  # Ignore padding/subword tokens
                pred_tags.append(id2label[p])
                gold_tags.append(id2label[l])
        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)
    
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [16]:
# Training configuration
# Adjust batch size based on GPU memory (16 works on Kaggle T4)

training_args = TrainingArguments(
    output_dir="./models/ner_kbbert_1177",
    
    # Training schedule
    num_train_epochs=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # Batch sizes (reduce if OOM)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    
    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    
    # Logging
    logging_steps=50,
    logging_first_step=True,
    report_to=[],  # Disable W&B etc.
    
    # Performance (enable on GPU)
    fp16=True,  # Mixed precision - faster training
    
    # Reproducibility
    seed=42,
)

# Data collator handles padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("Training configuration ready")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")

Training configuration ready
  Epochs: 4
  Learning rate: 2e-05
  Batch size: 16


### 3.3 Train the Model

Fine-tuning KB-BERT on ~880 training examples from 1177. Expected training time: ~5-10 minutes on Kaggle GPU.

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_featurized["train"],
    eval_dataset=ds_featurized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train!
print("Starting training...")
train_result = trainer.train()

# Print training summary
print(f"\nTraining completed!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

/tmp/ipykernel_58/3974991689.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,2.231700,0.364542,0.455882,0.476923,0.466165
2,0.629400,0.058428,0.847222,0.938462,0.890511
3,0.629400,0.023594,0.927536,0.984615,0.955224
4,0.032600,0.015049,0.954545,0.969231,0.961832


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Training completed!
  Total steps: 112
  Training loss: 0.3116


### 3.4 Evaluation

We evaluate the fine-tuned model on the held-out validation set and report per-class metrics.

In [19]:
# Final evaluation
eval_results = trainer.evaluate()

print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall:    {eval_results['eval_recall']:.4f}")
print(f"F1-Score:  {eval_results['eval_f1']:.4f}")
print("=" * 50)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


VALIDATION RESULTS
Precision: 0.9545
Recall:    0.9692
F1-Score:  0.9618


In [21]:
# Get detailed per-class breakdown
from seqeval.metrics import classification_report

# Run predictions on validation set
predictions, labels, _ = trainer.predict(ds_featurized["validation"])
predictions = np.argmax(predictions, axis=-1)

# Convert to label strings
true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    pred_tags = []
    gold_tags = []
    for p, l in zip(pred_seq, label_seq):
        if l != -100:
            pred_tags.append(id2label[p])
            gold_tags.append(id2label[l])
    true_predictions.append(pred_tags)
    true_labels.append(gold_tags)

# Print detailed report
print("\nPER-CLASS METRICS:")
print("=" * 60)
print(classification_report(true_labels, true_predictions, digits=4))


PER-CLASS METRICS:
                     precision    recall  f1-score   support

     body_structure     0.7500    0.8182    0.7826        11
   disorder_finding     1.0000    1.0000    1.0000        26
pharmaceutical_drug     1.0000    1.0000    1.0000        28

          micro avg     0.9545    0.9692    0.9618        65
          macro avg     0.9167    0.9394    0.9275        65
       weighted avg     0.9577    0.9692    0.9632        65



### 3.5 Save the Model

Save the fine-tuned model for later use in entity linking (Task 4).

In [22]:
import os
from datetime import datetime

# Create model directory with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
model_dir = f"./models/ner_kbbert_1177_{timestamp}"

# Save model and tokenizer
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

print(f"Model saved to: {model_dir}")
print(f"Contents: {os.listdir(model_dir)}")

Model saved to: ./models/ner_kbbert_1177_20260815_1424
Contents: ['tokenizer_config.json', 'vocab.txt', 'special_tokens_map.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'training_args.bin']


### 3.6 Test Entity Extraction

Let's test the model on a few example sentences to verify it extracts entities correctly.

In [23]:
from transformers import pipeline

# Create NER pipeline with our fine-tuned model
ner_pipeline = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # Merge B-/I- into single entities
    device=0 if trainer.args.fp16 else -1,  # Use GPU if available
)

# Test sentences (from 1177.se domain)
test_sentences = [
    "Patienten har diagnosen diabetes och tar metformin dagligen.",
    "Hon upplever smärta i höger knä och vänster axel.",
    "Läkaren ordinerade ibuprofen mot inflammationen.",
    "Symtom på stroke inkluderar domningar i ansiktet och svaghet i armar.",
]

print("ENTITY EXTRACTION EXAMPLES")
print("=" * 70)

for sentence in test_sentences:
    print(f"\nInput: {sentence}")
    entities = ner_pipeline(sentence)
    
    if entities:
        print("Entities found:")
        for ent in entities:
            print(f"  - '{ent['word']}' → {ent['entity_group']} (score: {ent['score']:.3f})")
    else:
        print("  No entities found")
    print("-" * 70)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ENTITY EXTRACTION EXAMPLES

Input: Patienten har diagnosen diabetes och tar metformin dagligen.
  No entities found
----------------------------------------------------------------------

Input: Hon upplever smärta i höger knä och vänster axel.
  No entities found
----------------------------------------------------------------------

Input: Läkaren ordinerade ibuprofen mot inflammationen.
  No entities found
----------------------------------------------------------------------

Input: Symtom på stroke inkluderar domningar i ansiktet och svaghet i armar.
  No entities found
----------------------------------------------------------------------


### 3.7 Summary

We have successfully fine-tuned KB-BERT for Swedish biomedical NER. The model identifies three entity types:

| Entity Type | Description | Examples |
|-------------|-------------|----------|
| `Disorder and Finding` | Diseases, symptoms, conditions | diabetes, smärta, stroke |
| `Pharmaceutical Drug` | Medications | metformin, ibuprofen |
| `Body Structure` | Anatomical locations | knä, axel, ansikte |

The model is now ready for entity linking (Task 4), where we will map extracted entities to ICD-10-SE codes.

In [27]:
#Debug
# Test on an ACTUAL example from the validation set
val_example = per_source_norm["1177"]["validation"][0]
print("Original sentence:", val_example["sentence"])
print("Entities:", val_example["entities"])

# Now test the pipeline on this exact sentence
result = ner_pipeline(val_example["sentence"])
print("Pipeline output:", result)

Original sentence: ( Demens ) innebär att man på olika sätt får svårt att minnas och att tolka sin omgivning.
Entities: [{'end': 10, 'start': 0, 'text': 'Demens', 'type': 0}]
Pipeline output: [{'entity_group': 'disorder_finding', 'score': np.float32(0.9778191), 'word': 'Demens', 'start': 2, 'end': 8}]


In [29]:
import torch

# Create NER pipeline with explicit device handling
ner_pipeline = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

# Also try without aggregation to see raw outputs
ner_pipeline_raw = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="none",  # See individual token predictions
    device=0 if torch.cuda.is_available() else -1,
)

# Test
test = "Patienten har diagnosen diabetes och tar metformin dagligen."
print("Raw predictions:")
for token in ner_pipeline_raw(test):
    if token["entity"] != "O":  # Only show non-O predictions
        print(f"  {token}")

Device set to use cuda:0
Device set to use cuda:0


Raw predictions:


In [33]:
# Direct model inference
import torch

model.eval()
test_sentence = "Patienten har diagnosen diabetes och tar metformin dagligen."

# Tokenize
inputs = tokenizer(test_sentence, return_tensors="pt", truncation=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)[0]

# Decode
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
for token, pred_id in zip(tokens, predictions):
    label = id2label[pred_id.item()]
    if label != "O":
        print(f"{token:20} → {label}")

Done
